# Clase 8 · ¿Cómo funciona un modelo de lenguaje?

ChatGPT, Gemini o Claude parecen conversar, resumir, traducir, opinar. Pero por dentro hacen **una sola cosa**, una y otra vez:

> **predecir cuál es la siguiente palabra.**

Hoy vamos a entender esa idea en tres pasos:

1. **Construir nuestro propio modelo de lenguaje**, con un `for` y un contador, a partir de 200 mil palabras de noticias.
2. **Mirar dentro de un modelo de lenguaje pequeño (SLM)** de verdad: ver qué palabras considera probables y cómo elige.
3. **Usar ese modelo para una tarea real de investigación**: anonimizar un testimonio, sin que el texto salga de nuestro entorno.

**Pregunta guía:** si un modelo de lenguaje solo repite lo que es probable en sus datos, ¿qué mundo nos devuelve?

## 1. Preparar el entorno

Activen la GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.

In [ ]:
import random
from collections import Counter

import nltk
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Parte 1 · Un modelo de lenguaje hecho a mano

## 2. El corpus: lo que va a "leer" nuestro modelo

Todo modelo de lenguaje aprende de un **corpus**: una colección grande de textos. El nuestro es **CESS-ESP**: unas 6.000 oraciones de noticias de la agencia EFE, de alrededor del año 2000.

In [ ]:
nltk.download("cess_esp")
from nltk.corpus import cess_esp

oraciones = cess_esp.sents()
print(len(oraciones))
print(oraciones[0])

Cada oración viene partida en tokens, pero con algunas marcas de los lingüistas que lo prepararon (`-Fpa-`, `*0*`, guiones bajos en `Electricité_de_France`). Esta función las limpia y junta todo en un solo texto.

No hace falta entender cada línea de la función. Lo importante es el resultado.

In [ ]:
def limpiar(oracion):
    tokens = [token for token in oracion if not token.startswith(("-F", "*0*"))]
    return " ".join(tokens).replace("_", " ")

texto = " ".join(limpiar(oracion) for oracion in oraciones)
palabras = texto.lower().split()

print(len(palabras))
print(texto[:300])

Son **197.618 palabras**. Pasamos todo a minúsculas para que "El" y "el" cuenten como la misma palabra.

### ¿Qué mundo cuenta este corpus?

Antes de entrenar nada, miremos de qué habla. `Counter` cuenta cuántas veces aparece cada palabra:

In [ ]:
frecuencias = Counter(palabras)

print(frecuencias["españa"])
print(frecuencias["ecuador"])

### Tu turno

Cuenten cuántas veces aparecen estas palabras y anoten los resultados:

`madrid`, `quito`, `hombre`, `mujer`, `indígenas`, `kichwa`, `minga`, `inmigrantes`.

Pista: pueden escribir una línea por palabra, o hacer una lista y recorrerla con un `for`.

In [ ]:
# escribe aquí

**Para comentar:** "kichwa" y "minga" aparecen **cero** veces. Para un modelo entrenado con este corpus, esas palabras **no existen**. ¿Qué pasa cuando le preguntamos algo sobre ellas?

## 3. Contar qué palabra viene después

Aquí está toda la idea de un modelo de lenguaje: para cada palabra, contamos **qué palabras aparecen justo después**. A cada par de palabras seguidas se le llama **bigrama**.

Para eso usamos un diccionario donde **cada palabra tiene su propio contador**:

In [ ]:
siguientes = {}

for palabra, despues in zip(palabras, palabras[1:]):
    if palabra not in siguientes:
        siguientes[palabra] = Counter()
    siguientes[palabra][despues] += 1

**Línea por línea**
- `palabras[1:]` es la misma lista, pero empezando desde la segunda palabra.
- `zip(palabras, palabras[1:])` junta cada palabra con la que viene después: ("el", "grupo"), ("grupo", "estatal"), ("estatal", "electricité")...
- `if palabra not in siguientes:` si es la primera vez que vemos esa palabra, le creamos un contador vacío.
- `siguientes[palabra][despues] += 1` le suma uno a la palabra que vino después.

Eso es todo: **acabamos de entrenar un modelo de lenguaje**. Veamos qué aprendió después de "el":

In [ ]:
siguientes["el"].most_common(10)

Convirtamos esos conteos en **probabilidades**: cuántas veces apareció cada palabra, dividido para el total.

In [ ]:
total = sum(siguientes["el"].values())

for despues, veces in siguientes["el"].most_common(10):
    print(despues, round(veces / total, 3))

Según nuestro modelo, después de "el" hay un 2,1 % de probabilidad de que venga "presidente" y un 1,5 % de que venga "gobierno". **Eso es lo que "sabe" un modelo de lenguaje: probabilidades.**

### Tu turno

Miren qué palabras vienen después de `"mujeres"` y después de `"inmigrantes"`.

In [ ]:
# escribe aquí

**Para comentar:** después de "inmigrantes", la palabra más probable es "ilegales". El modelo no tiene una opinión: solo aprendió **cómo hablaban de los inmigrantes las noticias con las que lo entrenamos**. Los modelos grandes funcionan igual, con mucho más texto.

## 4. Generar texto

Para "escribir", el modelo hace esto:
1. mira la palabra actual;
2. **sortea** la siguiente, dándole más chance a las más frecuentes;
3. repite con la nueva palabra.

`random.choices` hace el sorteo: `weights` le dice cuánto "pesa" cada opción.

In [ ]:
def generar(palabra, cuantas=20):
    texto_generado = [palabra]
    for vuelta in range(cuantas):
        opciones = siguientes[palabra]
        palabra = random.choices(list(opciones.keys()), weights=list(opciones.values()))[0]
        texto_generado.append(palabra)
    return " ".join(texto_generado)

generar("el")

### Tu turno

1. Corran la celda de arriba **varias veces**. ¿Por qué cambia el resultado?
2. Generen texto empezando con `"la"`, `"gobierno"` y `"mujer"`.
3. Prueben empezar con `"kichwa"`. ¿Qué pasa? ¿Por qué?

In [ ]:
# escribe aquí

## 5. Lo que le falta a nuestro modelo: el contexto

El texto suena a noticia, con palabras que "pegan" de a dos. Pero no tiene sentido. ¿Por qué?

Porque nuestro modelo **solo mira una palabra hacia atrás**. Si la frase es *"Las mujeres de la comunidad organizaron una..."*, nuestro modelo solo ve "una". Ya olvidó que hablábamos de mujeres y de comunidad.

Los modelos de lenguaje modernos (los *transformers*) resuelven esto con tres ideas:

- **Tokens:** no parten en palabras completas, sino en pedazos de palabras. Así pueden escribir palabras que nunca vieron completas.
- **Embeddings:** cada token se convierte en una lista de números que funciona como una "ubicación" en un mapa de significados. "Quito" queda cerca de "Cuenca"; "minga" cerca de "trabajo comunitario"... si es que apareció en sus datos.
- **Atención:** antes de predecir, el modelo **mira todas las palabras anteriores** y decide cuáles importan más. Para completar *"Las mujeres de la comunidad organizaron una..."*, presta atención a "organizaron" y "comunidad", no solo a "una".

Pero el objetivo final es **el mismo** que el de nuestro modelo de bigramas: **calcular la probabilidad de la siguiente palabra y sortear**.

# Parte 2 · Un modelo de lenguaje pequeño de verdad

## 6. Cargar el modelo

Usamos **Qwen2.5-1.5B-Instruct**, un modelo pequeño (SLM) de código abierto. Tiene 1.500 millones de **parámetros** (los números que el modelo ajustó al entrenar). Para comparar: los modelos grandes como GPT-4 o Claude tienen cientos de miles de millones.

Lo importante: **corre aquí, en Colab**. Lo que le escribamos no se envía a ninguna empresa.

La primera vez descarga unos 3 GB, así que tarda un poco.

In [ ]:
nombre_modelo = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizador = AutoTokenizer.from_pretrained(nombre_modelo)
modelo = AutoModelForCausalLM.from_pretrained(nombre_modelo, dtype="auto", device_map="auto")

**Línea por línea**
- `AutoTokenizer` carga el **tokenizador**: la parte que corta el texto en tokens.
- `AutoModelForCausalLM` carga el **modelo**. *Causal LM* significa justamente "modelo que predice la siguiente palabra".
- `dtype="auto"` usa números más livianos para que quepa en la memoria.
- `device_map="auto"` pone el modelo en la GPU si hay una.

### ¿Cómo ve el texto este modelo?

In [ ]:
frase = "La comunidad organizó una minga para arreglar el camino"

tokens = tokenizador.tokenize(frase)
print(tokens)
print(len(tokens))

El símbolo `Ġ` marca que el token empieza con un espacio. Fíjense cómo corta "minga": como no la vio mucho, la arma con pedazos.

### Tu turno

Tokenicen una frase en inglés, otra en español y otra en kichwa (o con palabras en kichwa que conozcan), más o menos del mismo largo. ¿Cuál usa más tokens?

Cada token cuesta tiempo, memoria y, en los servicios pagados, dinero. Una lengua que se parte en más pedazos es una lengua **más cara y peor representada**.

In [ ]:
# escribe aquí

## 7. Mirar adentro: las probabilidades de la siguiente palabra

Esta función le da un texto al modelo y muestra los tokens que considera más probables para continuar. Es exactamente lo mismo que hicimos con `siguientes["el"]`, pero ahora el modelo **mira todo el texto**, no solo la última palabra.

In [ ]:
def siguientes_probables(texto, cuantos=10):
    entrada = tokenizador(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo(**entrada)
    puntajes = salida.logits[0, -1]
    probabilidades = torch.softmax(puntajes.float(), dim=-1)
    mejores = torch.topk(probabilidades, cuantos)
    for probabilidad, numero in zip(mejores.values, mejores.indices):
        print(repr(tokenizador.decode(numero)), round(probabilidad.item(), 3))

siguientes_probables("Llegamos a Quito buscando")


**Línea por línea** (no hace falta memorizarlas, sí entender la idea)
- `tokenizador(...)` convierte el texto en números de tokens.
- `modelo(**entrada)` hace la predicción. El modelo devuelve un **puntaje** (*logit*) para **cada uno** de los ~150 mil tokens que conoce.
- `torch.softmax(...)` convierte esos puntajes en probabilidades que suman 1. Es lo mismo que hicimos al dividir para el total.
- `torch.topk(...)` se queda con los más probables.

### Tu turno

Comparen las predicciones para:
1. `"La mujer trabaja como"` y `"El hombre trabaja como"`
2. `"En la comunidad hablamos"`
3. `"Los inmigrantes son"`

¿Qué patrones ven? ¿De dónde salen?

In [ ]:
# escribe aquí

## 8. La temperatura: qué tan arriesgado es el sorteo

Al generar, el modelo sortea igual que nuestra función `generar`. La **temperatura** cambia cómo se reparten las probabilidades antes del sorteo:

- **temperatura baja** (0.1): casi siempre gana la opción más probable. Texto predecible y repetitivo.
- **temperatura 1**: el sorteo usa las probabilidades tal cual.
- **temperatura alta** (1.5 o más): las opciones raras ganan más chance. Texto creativo... o sin sentido.

In [ ]:
def continuar(texto, temperatura=1.0, maximo=40):
    entrada = tokenizador(texto, return_tensors="pt").to(modelo.device)
    salida = modelo.generate(**entrada, max_new_tokens=maximo, do_sample=True, temperature=temperatura)
    return tokenizador.decode(salida[0], skip_special_tokens=True)

print(continuar("En la comunidad hablamos", temperatura=0.1))

### Tu turno

Corran la misma frase con temperatura `0.1`, `1.0` y `2.0`, dos veces cada una. ¿Qué cambia? ¿Qué temperatura usarían para resumir una entrevista y cuál para escribir un poema?

In [ ]:
# escribe aquí

# Parte 3 · Una tarea real: anonimizar un testimonio

En investigación con personas, antes de compartir o analizar una entrevista hay que **quitar los datos que identifican** a quien habla. Hacerlo a mano con cientos de entrevistas toma semanas.

¿Puede ayudar un modelo de lenguaje? Y si lo usamos, ¿dónde debería correr?

Este testimonio es **ficticio**, pero tiene el tipo de datos que aparecen en entrevistas reales:

In [ ]:
testimonio = """Me llamo Rosa Quishpe, tengo 43 años y vivo en el barrio La Ferroviaria, en el sur de Quito. Llegué de Guamote en 2009 con mi esposo, Manuel Guamán, porque en la comunidad ya no alcanzaba la cosecha. Al principio trabajé limpiando casas en Cumbayá, donde la señora Patricia Andrade me pagaba quince dólares al día. Ahora vendo hierbas medicinales en el mercado de Chiriyacu. Mis hijos, Kevin y Mayra, estudian en la Unidad Educativa Manuela Cañizares. Soy la única dirigenta kichwa de la asociación de comerciantes del mercado, y eso a veces me trae problemas con el administrador, el señor Luis Toapanta. Si necesitan algo, mi número es 0998 123 456."""

Para conversar con el modelo usamos la **plantilla de chat**: el formato en el que fue entrenado para seguir instrucciones. Esta función se encarga de eso.

In [ ]:
def preguntar(instruccion, temperatura=0.2, maximo=400):
    mensajes = [{"role": "user", "content": instruccion}]
    entrada = tokenizador.apply_chat_template(
        mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(modelo.device)
    salida = modelo.generate(**entrada, max_new_tokens=maximo, do_sample=True, temperature=temperatura)
    respuesta = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizador.decode(respuesta, skip_special_tokens=True)

**Línea por línea**
- `mensajes` es la conversación: una lista con un mensaje del usuario (`"user"`).
- `apply_chat_template` le pone al texto las marcas especiales que el modelo espera ("aquí habla el usuario", "aquí responde el asistente").
- `modelo.generate(...)` genera la respuesta token por token, con el sorteo que ya conocemos.
- `salida[0][entrada["input_ids"].shape[1]:]` se queda solo con lo nuevo: la respuesta, sin repetir nuestra pregunta.

Usamos temperatura baja porque queremos precisión, no creatividad.

In [ ]:
instruccion = f"""Anonimiza el siguiente testimonio. Reemplaza los nombres de personas por [PERSONA],
los lugares por [LUGAR], las instituciones por [INSTITUCIÓN] y los números de teléfono por [TELÉFONO].
No cambies nada más del texto.

Testimonio:
{testimonio}"""

anonimizado = preguntar(instruccion)
print(anonimizado)

## 9. Verificar: nunca confiar sin revisar

Un modelo de lenguaje no "sabe" qué es un dato sensible: predice qué texto es probable después de nuestra instrucción. Así que **revisamos**. Hagamos una lista de los datos que no deberían aparecer y comprobemos si siguen ahí:

In [ ]:
datos_sensibles = [
    "Rosa Quishpe", "Manuel Guamán", "Patricia Andrade", "Luis Toapanta", "Kevin", "Mayra",
    "La Ferroviaria", "Guamote", "Cumbayá", "Chiriyacu", "Manuela Cañizares", "0998 123 456",
]

for dato in datos_sensibles:
    if dato in anonimizado:
        print("SE FILTRÓ:", dato)

**Para comentar:**
- ¿Se le escapó algún dato? ¿Cuál?
- ¿Cambió algo más del texto que no le pedimos? Comparen con el original.
- Lean esta frase: *"Soy la única dirigenta kichwa de la asociación de comerciantes del mercado"*. No tiene ningún nombre... ¿se puede identificar a Rosa con ella?
- Corran la anonimización otra vez. ¿Sale igual? ¿Qué implica eso para un proceso de investigación?

### Tu turno

Elijan **una** de estas opciones:

1. **Mejorar la instrucción:** reescríbanla para que el modelo también trate la edad, el año y los datos que identifican de forma indirecta. Vuelvan a verificar.
2. **Otra tarea:** pídanle que resuma el testimonio en tres frases, o que extraiga los temas principales con **una cita textual** que respalde cada tema. Después comprueben con `in` si las citas aparecen de verdad en el testimonio. ¿Inventó alguna?

In [ ]:
# escribe aquí